# Region Properties Analysis with SAM Refinement
## Extract morphological features from SAM-refined segmentations

This notebook:
1. Loads SAM-refined segmentation coordinates
2. Computes region properties (area, perimeter, eccentricity, etc.)
3. Saves results to CSV for downstream analysis

In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from skimage import measure
from skimage.draw import polygon
import matplotlib.pyplot as plt
from tqdm import tqdm

print('Libraries loaded successfully!')

## Configuration

In [ ]:
# Paths
sam_refined_dir = Path('results/sam_refined/coordinates')
output_dir = Path('results/region_properties')
output_dir.mkdir(parents=True, exist_ok=True)

# Which coordinate space to use
# Options: 'resized', 'original_patch', 'wsi'
coordinate_space = 'original_patch'  # Use original patch coordinates

# Which segmentation to use
# Options: 'sam', 'yolo'
segmentation_type = 'sam'  # Use SAM refined segmentations

print(f'SAM refined directory: {sam_refined_dir}')
print(f'Output directory: {output_dir}')
print(f'Coordinate space: {coordinate_space}')
print(f'Segmentation type: {segmentation_type}')

## Load SAM Refined Data

In [ ]:
def load_sam_refined_data(json_path):
    """Load SAM refined JSON file."""
    with open(json_path, 'r') as f:
        data = json.load(f)
    return data

def get_segmentation_coords(detection, seg_type='sam', coord_space='original_patch'):
    """
    Extract segmentation coordinates from detection.
    
    Args:
        detection: Detection dictionary from SAM refined JSON
        seg_type: 'sam' or 'yolo'
        coord_space: 'resized', 'original_patch', or 'wsi'
    
    Returns:
        List of polygons (each polygon is a list of [x, y] coordinates)
    """
    key = f'{seg_type}_segmentation_{coord_space}'
    return detection.get(key, [])

# Load all SAM refined JSON files
json_files = sorted(sam_refined_dir.glob('*_sam_refined.json'))
print(f'Found {len(json_files)} SAM refined JSON files')

if len(json_files) > 0:
    # Show example of first file
    example_data = load_sam_refined_data(json_files[0])
    print(f"\nExample file: {json_files[0].name}")
    print(f"Patch: {example_data['patch_name']}")
    print(f"Number of detections: {example_data['num_detections']}")
    print(f"Patch info: {example_data['patch_info']}")

## Compute Region Properties

In [ ]:
def polygon_to_mask(polygons, shape):
    """
    Convert polygon coordinates to binary mask.
    
    Args:
        polygons: List of polygons, each polygon is [[x1,y1], [x2,y2], ...]
        shape: (height, width) of output mask
    
    Returns:
        Binary mask (numpy array)
    """
    mask = np.zeros(shape, dtype=np.uint8)
    
    for polygon_coords in polygons:
        if len(polygon_coords) > 0:
            # Extract x and y coordinates
            coords = np.array(polygon_coords)
            if coords.shape[0] < 3:  # Need at least 3 points for a polygon
                continue
            
            x_coords = coords[:, 0]
            y_coords = coords[:, 1]
            
            # Fill polygon
            rr, cc = polygon(y_coords, x_coords, shape)
            mask[rr, cc] = 1
    
    return mask

def compute_region_properties(mask, intensity_image=None):
    """
    Compute morphological properties from binary mask.
    
    Args:
        mask: Binary mask
        intensity_image: Optional intensity image for intensity-based properties
    
    Returns:
        Dictionary of properties
    """
    # Label connected components
    labeled = measure.label(mask)
    
    if labeled.max() == 0:
        return None
    
    # Get properties for the largest region (assuming single object)
    regions = measure.regionprops(labeled, intensity_image=intensity_image)
    
    if len(regions) == 0:
        return None
    
    # Get largest region
    region = max(regions, key=lambda r: r.area)
    
    properties = {
        'area': region.area,
        'perimeter': region.perimeter,
        'eccentricity': region.eccentricity,
        'solidity': region.solidity,
        'extent': region.extent,
        'major_axis_length': region.major_axis_length,
        'minor_axis_length': region.minor_axis_length,
        'orientation': region.orientation,
        'centroid_y': region.centroid[0],
        'centroid_x': region.centroid[1],
        'bbox_min_y': region.bbox[0],
        'bbox_min_x': region.bbox[1],
        'bbox_max_y': region.bbox[2],
        'bbox_max_x': region.bbox[3],
    }
    
    # Add circularity (4π × area / perimeter²)
    if region.perimeter > 0:
        properties['circularity'] = (4 * np.pi * region.area) / (region.perimeter ** 2)
    else:
        properties['circularity'] = 0
    
    return properties

## Process All Detections

In [ ]:
# Store results
results = []

# Process each JSON file
for json_path in tqdm(json_files, desc='Processing patches'):
    data = load_sam_refined_data(json_path)
    
    patch_name = data['patch_name']
    patch_info = data['patch_info']
    
    # Determine shape based on coordinate space
    if coordinate_space == 'original_patch':
        shape = (patch_info['original_height'], patch_info['original_width'])
    elif coordinate_space == 'resized':
        # Assuming resized to 1024x768 (standard for SAM)
        shape = (768, 1024)
    else:  # wsi
        # For WSI, we'll use original patch shape (WSI coords are absolute)
        shape = (patch_info['original_height'], patch_info['original_width'])
    
    # Process each detection
    for detection in data['detections']:
        detection_id = detection['detection_id']
        class_name = detection['class']
        confidence = detection['confidence']
        
        # Get segmentation coordinates
        polygons = get_segmentation_coords(detection, segmentation_type, coordinate_space)
        
        if not polygons or len(polygons) == 0:
            continue
        
        # Create binary mask
        mask = polygon_to_mask(polygons, shape)
        
        if mask.sum() == 0:  # Empty mask
            continue
        
        # Compute properties
        props = compute_region_properties(mask)
        
        if props is None:
            continue
        
        # Add metadata
        result = {
            'patch_name': patch_name,
            'detection_id': detection_id,
            'class': class_name,
            'confidence': confidence,
            'segmentation_type': segmentation_type,
            'coordinate_space': coordinate_space,
            **props
        }
        
        # Add WSI coordinates if available
        if 'wsi_xmin' in patch_info:
            result['wsi_xmin'] = patch_info['wsi_xmin']
            result['wsi_ymin'] = patch_info['wsi_ymin']
            result['wsi_xmax'] = patch_info['wsi_xmax']
            result['wsi_ymax'] = patch_info['wsi_ymax']
        
        results.append(result)

print(f'\nProcessed {len(results)} detections successfully')

## Save Results

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(results)

# Save to CSV
output_file = output_dir / f'region_properties_{segmentation_type}_{coordinate_space}.csv'
df.to_csv(output_file, index=False)

print(f'\nResults saved to: {output_file}')
print(f'\nDataFrame shape: {df.shape}')
print(f'\nColumn names:')
print(df.columns.tolist())
print(f'\nFirst few rows:')
df.head()

## Summary Statistics

In [ ]:
# Summary by class
print('\n=== Summary by Class ===')
for class_name in df['class'].unique():
    class_df = df[df['class'] == class_name]
    print(f'\n{class_name.upper()} (n={len(class_df)})')
    print(f"  Area: {class_df['area'].mean():.2f} ± {class_df['area'].std():.2f}")
    print(f"  Perimeter: {class_df['perimeter'].mean():.2f} ± {class_df['perimeter'].std():.2f}")
    print(f"  Eccentricity: {class_df['eccentricity'].mean():.3f} ± {class_df['eccentricity'].std():.3f}")
    print(f"  Solidity: {class_df['solidity'].mean():.3f} ± {class_df['solidity'].std():.3f}")
    print(f"  Circularity: {class_df['circularity'].mean():.3f} ± {class_df['circularity'].std():.3f}")

# Describe all numeric columns
print('\n=== Detailed Statistics ===')
df.describe()

## Visualize Distributions

In [ ]:
# Plot key morphological features
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.ravel()

features = ['area', 'perimeter', 'eccentricity', 'solidity', 'circularity', 'major_axis_length']
colors = {'crypt': 'blue', 'gland': 'red'}

for idx, feature in enumerate(features):
    ax = axes[idx]
    
    for class_name in df['class'].unique():
        class_df = df[df['class'] == class_name]
        ax.hist(class_df[feature], bins=30, alpha=0.5, 
                label=class_name, color=colors.get(class_name, 'gray'))
    
    ax.set_xlabel(feature.replace('_', ' ').title())
    ax.set_ylabel('Count')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / f'morphological_distributions_{segmentation_type}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'Distribution plot saved to: {output_dir / f"morphological_distributions_{segmentation_type}.png"}')

## Compare SAM vs YOLO (Optional)

In [ ]:
# If you want to compare SAM vs YOLO segmentations
# Process both and create comparison

def compare_sam_vs_yolo():
    """Compare SAM refined vs YOLO segmentations."""
    
    comparison_results = []
    
    for json_path in tqdm(json_files[:5], desc='Comparing SAM vs YOLO'):  # First 5 for demo
        data = load_sam_refined_data(json_path)
        
        patch_info = data['patch_info']
        shape = (patch_info['original_height'], patch_info['original_width'])
        
        for detection in data['detections']:
            detection_id = detection['detection_id']
            
            # Get both SAM and YOLO segmentations
            sam_polys = get_segmentation_coords(detection, 'sam', coordinate_space)
            yolo_polys = get_segmentation_coords(detection, 'yolo', coordinate_space)
            
            if not sam_polys or not yolo_polys:
                continue
            
            # Create masks
            sam_mask = polygon_to_mask(sam_polys, shape)
            yolo_mask = polygon_to_mask(yolo_polys, shape)
            
            if sam_mask.sum() == 0 or yolo_mask.sum() == 0:
                continue
            
            # Compute IoU
            intersection = np.logical_and(sam_mask, yolo_mask).sum()
            union = np.logical_or(sam_mask, yolo_mask).sum()
            iou = intersection / union if union > 0 else 0
            
            # Compute properties for both
            sam_props = compute_region_properties(sam_mask)
            yolo_props = compute_region_properties(yolo_mask)
            
            if sam_props and yolo_props:
                comparison_results.append({
                    'patch_name': data['patch_name'],
                    'detection_id': detection_id,
                    'class': detection['class'],
                    'iou': iou,
                    'sam_area': sam_props['area'],
                    'yolo_area': yolo_props['area'],
                    'area_diff_pct': 100 * (sam_props['area'] - yolo_props['area']) / yolo_props['area'],
                    'sam_circularity': sam_props['circularity'],
                    'yolo_circularity': yolo_props['circularity'],
                })
    
    if comparison_results:
        comp_df = pd.DataFrame(comparison_results)
        comp_df.to_csv(output_dir / 'sam_vs_yolo_comparison.csv', index=False)
        
        print(f'\n=== SAM vs YOLO Comparison ===')
        print(f"Mean IoU: {comp_df['iou'].mean():.3f} ± {comp_df['iou'].std():.3f}")
        print(f"Mean area difference: {comp_df['area_diff_pct'].mean():.1f}% ± {comp_df['area_diff_pct'].std():.1f}%")
        
        return comp_df
    
    return None

# Run comparison
comp_df = compare_sam_vs_yolo()
if comp_df is not None:
    display(comp_df.head())